# Model retries and fallback

Retry repeats a failed call to the same model. Fallback tries alternative model objects in order. Exception filters distinguish retryable failures; backoff controls the waiting time between attempts.

[Model retries and fallback](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-retry)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.faults import FailFirstModelCalls, FailPrimaryModel
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware, ModelRetryMiddleware
from model_config import get_model, run_config
from lab.faults import ScriptedModel
from langchain_core.messages import AIMessage


In [ ]:
# Execute
model = get_model()
fault = FailFirstModelCalls(failures=2)
agent = create_agent(
    model=model,
    middleware=[
        ModelRetryMiddleware(max_retries=2, initial_delay=0.1, jitter=False, retry_on=(ConnectionError,), backoff_factor=2, max_delay=0.2, on_failure="error"),
        TraceMiddleware("03_retry", pretty=True), fault,
    ],
)
retry_result = agent.invoke({"messages": [("user", prompt("hello"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(retry_result["messages"], pretty=True)
print("Attempts:", fault.attempts)

### Fallback

The primary model raises a connection error and the fallback handles the request. Separate objects using the same endpoint demonstrate routing; surviving an endpoint outage requires a fallback on another available endpoint.

In [ ]:
# Execute
primary = get_model()
fallback = get_model("fallback")
agent = create_agent(
    model=primary,
    middleware=[
        ModelFallbackMiddleware(fallback),
        TraceMiddleware("03_fallback", pretty=True),
        FailPrimaryModel(primary),
    ],
)
fallback_result = agent.invoke({"messages": [("user", prompt("hello"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(fallback_result["messages"], pretty=True)

### A retry budget can also be exhausted

In [ ]:
agent = create_agent(
    model=model,
    middleware=[
        ModelRetryMiddleware(max_retries=1, initial_delay=0.1, jitter=False, on_failure="error"),
        FailFirstModelCalls(failures=100),
    ],
)
with expect_error(ConnectionError):
    agent.invoke({"messages": [("user", prompt("hello"))]}, config=run_config())

### Continue after exhausted retries

An exhausted retry can either propagate its exception or return a final error response. A custom failure callback can supply that response. Fallback middleware accepts several alternatives and tries them in order until one succeeds.

In [ ]:
# Execute
continued_agent = create_agent(model=ScriptedModel(responses=[AIMessage(content="Unreachable")]), middleware=[
    ModelRetryMiddleware(max_retries=1, initial_delay=0, on_failure=lambda error: "The model endpoint is temporarily unavailable."),
    FailFirstModelCalls(failures=100),
])
continued_result = continued_agent.invoke({"messages": [("user", "Hello")]}, config=run_config())

In [ ]:
# Inspect output
show_messages(continued_result["messages"], pretty=True)
assert "temporarily unavailable" in continued_result["messages"][-1].content